# Exercise 5: RoPE（旋转位置编码）

**Goal:** 实现 Qwen3 用的旋转位置编码，这是 Phase 2 里最难的一个

**数学原理：**

对于位置 $m$，维度对 $i$，旋转角度为：
$$\theta_i = m / 10000^{2i / d}$$

对每对 $(x_0, x_1)$ 做二维旋转：
$$x_0' = x_0 \cos\theta - x_1 \sin\theta$$
$$x_1' = x_0 \sin\theta + x_1 \cos\theta$$

**简化写法（rotate_half trick）：**
$$\text{RoPE}(x) = x \cdot \cos + \text{rotate\_half}(x) \cdot \sin$$
其中 `rotate_half` 把 head_dim 从中间切开，后半取负放前面：
$$\text{rotate\_half}([x_1, x_2]) = [-x_2, x_1]$$

**写之前先在脑子里回答：**
- 位置编码为什么加在 Q 和 K 上，不加在 V 上？
- 为什么是 head_dim 而不是 hidden_dim？
- cos/sin 怎么 broadcast 到 `(B, n_heads, T, head_dim)` 的 shape？

In [ ]:
import torch
import torch.nn as nn

In [ ]:
def precompute_freqs(head_dim: int, max_seq_len: int, base: float = 10000.0):
    """
    预计算 cos/sin 表
    返回: cos, sin，shape 均为 (max_seq_len, head_dim)
    """
    # YOUR CODE HERE
    pass


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """
    沿最后一维从中间切开，返回 [-x2, x1]
    输入/输出 shape: (..., head_dim)
    """
    # YOUR CODE HERE
    pass


def apply_rope(
    q: torch.Tensor,    # (B, T, n_heads, head_dim)
    k: torch.Tensor,    # (B, T, n_kv_heads, head_dim)
    cos: torch.Tensor,  # (T, head_dim)
    sin: torch.Tensor,  # (T, head_dim)
) -> tuple:
    # YOUR CODE HERE
    # 提示: cos/sin 需要 unsqueeze 才能 broadcast 到 batch 和 heads 维
    pass

In [ ]:
# Verification
torch.manual_seed(0)
B, T, n_heads, head_dim = 2, 6, 4, 32

q = torch.randn(B, T, n_heads, head_dim)
k = torch.randn(B, T, n_heads, head_dim)

cos, sin = precompute_freqs(head_dim, max_seq_len=T)
q_rot, k_rot = apply_rope(q, k, cos, sin)

print("q_rot shape:", q_rot.shape)  # 应该是 (2, 6, 4, 32)
print("k_rot shape:", k_rot.shape)

# 旋转不改变向量的模长
q_norm_before = q.norm(dim=-1).mean().item()
q_norm_after  = q_rot.norm(dim=-1).mean().item()
print(f"q norm before: {q_norm_before:.4f}, after: {q_norm_after:.4f}  (应该近似相等)")
assert abs(q_norm_before - q_norm_after) < 0.01, "norm 变化太大，检查你的旋转实现"
print("PASS")